> **참고용 사본 (reference copy).** 원본 노트북에서 출력, 실행 횟수, 로컬 경로, 키 값을 제거한 사본입니다. 비공개 원천 데이터가 필요해 이 저장소만으로는 실행되지 않으며, 코드 구조와 분석 절차를 보여주는 용도입니다.

*Reference copy of a private notebook: outputs and paths removed; requires non-public data to run.*


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
adj_dps         = pd.read_csv("./input/수정DPS.csv", index_col=0, parse_dates=True)
adj_close       = pd.read_csv("./input/수정종가.csv", index_col=0, parse_dates=True)
total_adj_close = pd.read_csv("./input/현금배당반영수정주가.csv", index_col=0, parse_dates=True)
is_kospi_raw    = pd.read_csv("./input/시장구분.csv", index_col=0, parse_dates=True)

adj_dps         = adj_dps.drop(columns=['인트로메딕', '파멥신'], errors='ignore')
adj_close       = adj_close.drop(columns=['인트로메딕', '파멥신'], errors='ignore')
total_adj_close = total_adj_close.drop(columns=['인트로메딕', '파멥신'], errors='ignore')
is_kospi_raw    = is_kospi_raw.drop(columns=['인트로메딕', '파멥신'], errors='ignore')

In [ ]:
adj_dps_shifted         = adj_dps.resample('ME').last().shift(17).apply(pd.to_numeric, errors="coerce")
adj_close_monthly       = adj_close.resample('ME').last()
total_adj_close_monthly = total_adj_close.resample('ME').last()
is_kospi                = (is_kospi_raw == 'KSE').astype(int).resample('ME').last()

monthly_returns         = adj_close_monthly.pct_change()
total_monthly_returns   = total_adj_close_monthly.pct_change()

In [ ]:
def winsorize_cross_section(df, lower_q=0.01, upper_q=0.99):
    lower = df.quantile(lower_q, axis=1)
    upper = df.quantile(upper_q, axis=1)
    return df.clip(lower=lower, upper=upper, axis=0)

monthly_returns = winsorize_cross_section(adj_close_monthly.pct_change())
total_monthly_returns = winsorize_cross_section(total_adj_close_monthly.pct_change())

In [ ]:
div_yield   = adj_dps_shifted / adj_close_monthly

In [ ]:
# 1) 원하는 연도 범위
start_year = 2008
end_year = 2026

# 2) 매년 6월 30일 생성
selection_dates = pd.to_datetime([
    f"{year}-06-30" for year in range(start_year, end_year + 1)
])

selection_dates

In [ ]:
import pandas as pd
import numpy as np

q5_returns_list = []

for i in range(len(selection_dates) - 1):
    start_date = selection_dates[i]
    end_date   = selection_dates[i + 1]

    # 배당수익률 > 0 인 종목만
    dy = div_yield.loc[start_date]
    dy = dy[dy > 0]

    # KOSPI 종목만으로 80% breakpoint 계산
    dy_kospi = dy[is_kospi.loc[start_date][dy.index] == 1]
    bp80 = dy_kospi.quantile(0.8)

    # 전체 종목 중 Q5(상위 20%) 선택
    selected_stocks = dy[dy > bp80].index

    # 월별 평균 수익률 계산 (리밸런싱 다음 시점부터)
    ret_price = monthly_returns.loc[start_date:end_date, selected_stocks].iloc[1:].mean(axis=1)
    ret_total = total_monthly_returns.loc[start_date:end_date, selected_stocks].iloc[1:].mean(axis=1)

    # 하나의 데이터프레임으로 정리
    temp = pd.DataFrame({
        "RET": ret_price,
        "TOTAL_RET": ret_total
    })

    q5_returns_list.append(temp)

portfolio_ret = pd.concat(q5_returns_list).sort_index()
portfolio_ret.tail()

In [ ]:
(1+portfolio_ret).cumprod().plot()

---
#### **Abnomal Return**

In [ ]:
import pandas as pd

factor = pd.read_csv("./input/Factor.csv", index_col=0, parse_dates=True)
factor.tail()

In [ ]:
rf = (1 + factor['RF'] / 100) ** (1/12) - 1
factor = factor.pct_change(fill_method=None)
factor['RF'] = rf
factor['MKT'] = factor['KOSPI'] - factor['RF']

In [ ]:
factor.tail()

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

# 0) 준비: LS 초과수익(월말)과 요인 정렬
ls_excess = (portfolio_ret['RET'] - factor['RF']).dropna()

F = factor.loc[ls_excess.index, ['MKT', 'SMB', 'HML', 'MOM']].copy()
data = pd.concat([ls_excess.rename('y'), F], axis=1).dropna()

y = data['y']
X = sm.add_constant(data[['MKT', 'SMB', 'HML', 'MOM']])

# 1) Rolling 회귀
W = 60
rol = RollingOLS(endog=y, exog=X, window=W).fit()

# 2) 미래정보유출 방지: t월 수익률은 t-1까지 추정한 계수 사용
params_lag = rol.params.shift(1)

# 중요: min_count=1 넣기
fitted_lag = (X * params_lag).sum(axis=1, min_count=1)
carhart4_adj_ret = (y - fitted_lag).dropna()

# 비교용: in-window residual
fitted_in = (X * rol.params).sum(axis=1, min_count=1)
carhart4_adj_ret_rolling_inwindow = (y - fitted_in).dropna()

carhart4_adj_ret.tail()

In [ ]:
(carhart4_adj_ret+1).cumprod().plot()

In [ ]:
portfolio_ret['ADJ_RET'] = carhart4_adj_ret
portfolio_ret.tail()

---
#### **Hedge Portfolio**

In [ ]:
import pandas as pd
from statsmodels.regression.rolling import RollingOLS

# ret_excess = portfolio_ret['TOTAL_RET'] - factor['RF']      # beta 추정용
# total_ret  = portfolio_ret['TOTAL_RET'] - factor['RF']      # 실제 헤지할 총수익률

ret_excess = portfolio_ret['RET'] - factor['RF']      # beta 추정용
total_ret  = portfolio_ret['RET'] - factor['RF']      # 실제 헤지할 총수익률

data = pd.concat([
    ret_excess.rename('ret_excess'),
    total_ret.rename('total_ret'),
    factor[['MKT', 'SMB', 'HML', 'MOM']]
], axis=1).dropna(subset=['ret_excess', 'MKT', 'SMB', 'HML', 'MOM'])

y_ret = data['ret_excess']
X = data[['MKT', 'SMB', 'HML', 'MOM']]

W = 36
rol = RollingOLS(y_ret, X, window=W).fit()
params_lag = rol.params.shift(1)

hedge_data = pd.concat([
    total_ret.rename('total_ret'),
    factor[['MKT', 'SMB', 'HML', 'MOM']]
], axis=1).dropna()

X_hedge = hedge_data[['MKT', 'SMB', 'HML', 'MOM']]
params_lag = params_lag.reindex(X_hedge.index)

fitted = (X_hedge * params_lag).sum(axis=1, min_count=1)
carhart4_hedged_total_ret = hedge_data['total_ret'] - fitted

In [ ]:
(carhart4_hedged_total_ret.dropna()+1).cumprod().plot()

In [ ]:
portfolio_ret['HEDGED_RET'] = carhart4_hedged_total_ret
portfolio_ret.tail()

In [ ]:
(portfolio_ret[['TOTAL_RET', 'HEDGED_RET']].dropna()+1).cumprod().plot()

In [ ]:
nav = (portfolio_ret + 1).cumprod()

mom_12_1 = (nav / nav.shift(11) - 1).shift(1)
mom_12_1.dropna()

In [ ]:
portfolio_ret['RET_MOM_12_2'] = mom_12_1['RET']
portfolio_ret['TOTAL_RET_MOM_12_2'] = mom_12_1['TOTAL_RET']
portfolio_ret['ADJ_RET_MOM_12_2'] = mom_12_1['ADJ_RET']
portfolio_ret['HEDGED_RET_MOM_12_2'] = mom_12_1['HEDGED_RET']
portfolio_ret['ABS_TOTAL_RET'] = portfolio_ret['TOTAL_RET'].abs()

In [ ]:
portfolio_ret.dropna()

In [ ]:
portfolio_ret.columns

In [ ]:
portfolio_ret.dropna().to_csv("./output/portfolio_ret.csv")